# A10: Expanding the usable population with more financial data (Tanner Edwards)

Personal EDA notebook. TMDB alone gives 5,766 usable films (Matt's contract population). This notebook adds financial data from other sources to grow that number.

| Source | Adds | Match to TMDB by |
|---|---|---|
| The Numbers (`data/raw/the_numbers/`) | Budget, **worldwide** gross, pre-release fields | Title + year (matcher below) |
| Wikidata snapshot (`data/raw/wikidata/`, committed) | Budget and box office in USD | TMDB id (exact) |
| Box Office Mojo (`data/raw/boxofficemojo/`) | Domestic gross only (no budget): evidence of a US theatrical run | Title + year |
| RT stefanoleone992 (`data/raw/stefanoleone992/`) | `actors`, `production_company` | Title + year |

**Precedence for budget and revenue:** TMDB → The Numbers → Wikidata (first non-missing value wins). Each film records which source each value came from.

Merge only. No outcome analysis or trimming yet. Solid results move to `A10-master.ipynb`.

## 1. Setup

In [ ]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
YEARS = (2000, 2023)          # contract window
BUDGET_FLOOR = 1_000_000      # contract floor

## 2. TMDB

All released, non-adult films (one row per id). Matching uses a wider year window (1998–2025) so films near the edges can still match; the contract window is applied after.

In [ ]:
t = pd.read_csv(RAW / "TMDB_movie_dataset_v11.csv", low_memory=False)
t["release_dt"] = pd.to_datetime(t["release_date"], errors="coerce")
t["year"] = t["release_dt"].dt.year
tmdb = t[t["status"].eq("Released") & ~t["adult"] & t["year"].between(1998, 2025)].drop_duplicates("id").copy()

# Sanity check: Matt's contract population is reproduced exactly
pop = tmdb[tmdb["year"].between(*YEARS) & tmdb["budget"].ge(BUDGET_FLOOR) & tmdb["revenue"].gt(0)]
assert len(pop) == 5766, len(pop)
print(f"TMDB candidates for matching: {len(tmdb):,}; contract population: {len(pop):,}")

## 3. Load the other sources

In [ ]:
tn = pd.read_csv(RAW / "the_numbers/Top Movies (Cleaned Data).csv", low_memory=False)
tn["year"] = pd.to_datetime(tn["Release Date"], errors="coerce").dt.year
tn = tn[tn["year"].between(*YEARS)].reset_index(drop=True)

wd = pd.read_csv(RAW / "wikidata/wikidata_film_financials_2026-11-21.csv")
wd = wd.rename(columns={"tmdb": "id"}).drop_duplicates("id")

bom_daily = pd.read_csv(RAW / "boxofficemojo/revenues_per_day.csv.gz", parse_dates=["date"])
# `id` in this file is per ROW (film-day), not per film: collapse to one row per title
bom = (bom_daily.groupby("title")
       .agg(first_date=("date", "min"), domestic_gross=("revenue", "sum"),
            days=("date", "nunique"), max_theaters=("theaters", "max"), distributor=("distributor", "first"))
       .reset_index())
bom["year"] = bom["first_date"].dt.year

sl = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_movies.csv", low_memory=False)
sl["year"] = pd.to_datetime(sl["original_release_date"], errors="coerce").dt.year

pd.Series({"The Numbers (2000-23)": len(tn), "Wikidata": len(wd), "Box Office Mojo titles": len(bom),
           "RT stefanoleone992": len(sl)}, name="rows").to_frame()

## 4. Title + year matcher

1. **Normalize titles:** strip accents, lowercase, `&` → `and`, drop a leading *The / A / An*, keep letters and digits only.
2. **Candidates:** same normalized title as TMDB's `title` **or** `original_title`, release year within ±2.
3. **Pick the best candidate** per source film: budget agrees (±15%), then closest year, then closest runtime, then most TMDB votes (votes only break ties for identity; they're never a model input).
4. **Year differences need corroboration:** if years differ, runtime must be within 10 minutes or the budget must agree.
5. **One-to-one only:** a TMDB film claimed by two source films is dropped from both.

In [ ]:
def norm_title(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str).map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode())
    s = s.str.lower().str.replace("&", " and ", regex=False)
    s = s.str.replace(r"^(the|a|an)\s+", "", regex=True)
    return s.str.replace(r"[^a-z0-9]", "", regex=True)


tmdb_keys = pd.concat([
    tmdb.assign(key=norm_title(tmdb["title"])),
    tmdb.assign(key=norm_title(tmdb["original_title"])),
]).drop_duplicates(["id", "key"])
tmdb_keys = tmdb_keys.loc[tmdb_keys["key"].ne(""), ["id", "key", "year", "runtime", "budget", "vote_count"]]


def match_to_tmdb(src, title_col, runtime_col=None, budget_col=None, name="src"):
    """Return src_row -> TMDB id, one-to-one, with match diagnostics."""
    s = pd.DataFrame({"src_row": np.arange(len(src)), "key": norm_title(src[title_col]), "src_year": src["year"]})
    s["src_runtime"] = src[runtime_col] if runtime_col else np.nan
    s["src_budget"] = src[budget_col] if budget_col else np.nan
    m = s.merge(tmdb_keys, on="key")
    m["year_diff"] = (m["src_year"] - m["year"]).abs()
    m = m[m["year_diff"] <= 2]
    m["runtime_diff"] = (m["src_runtime"] - m["runtime"]).abs()
    m["budget_agrees"] = m["budget"].gt(0) & ((m["budget"] / m["src_budget"] - 1).abs() < 0.15)
    m = (m.sort_values(["src_row", "budget_agrees", "year_diff", "runtime_diff", "vote_count"],
                       ascending=[True, False, True, True, False], na_position="last")
          .drop_duplicates("src_row"))
    m = m[m["year_diff"].eq(0) | m["runtime_diff"].le(10) | m["budget_agrees"]]
    m = m[~m["id"].duplicated(keep=False)]
    print(f"{name:<12} matched {len(m):,} of {len(src):,} ({len(m) / len(src):.0%}); "
          f"year diff counts {m['year_diff'].value_counts().sort_index().to_dict()}")
    return m[["src_row", "id", "year_diff", "runtime_diff", "budget_agrees"]]


tn_match = match_to_tmdb(tn, "Movie Name", "Running Time (minutes)", "Production Budget (USD)", "The Numbers")
bom_match = match_to_tmdb(bom, "title", name="Box Office Mojo")
sl_match = match_to_tmdb(sl, "movie_title", "runtime", name="RT sl")

both = tn_match.merge(tmdb[["id", "budget"]], on="id")
both = both[both["budget"].gt(0)]
print(f"\nCheck: where TMDB and The Numbers both have a budget, they agree (±15%) for "
      f"{both['budget_agrees'].mean():.0%} of {len(both):,} matched films")

## 5. Attach the sources to TMDB

In [ ]:
def attach(base, src, match, prefix, cols):
    part = src.iloc[match["src_row"]][cols].set_axis(match["id"].values)
    part.columns = [f"{prefix}_{c}" for c in cols]
    part[f"{prefix}_year_diff"] = match["year_diff"].values
    return base.merge(part, left_on="id", right_index=True, how="left")


tn_cols = ["Movie Name", "Release Date", "Production Budget (USD)", "Worldwide Gross (USD)", "Domestic Gross (USD)",
           "MPAA Rating", "Source", "Creative Type", "Production Method", "Franchise",
           "Production/Financing Companies", "Running Time (minutes)", "Movie URL"]
films = tmdb.copy()
films = attach(films, tn, tn_match, "tn", tn_cols)
films = attach(films, bom, bom_match, "bom", ["title", "first_date", "domestic_gross", "max_theaters"])
films = attach(films, sl, sl_match, "sl", ["movie_title", "actors", "production_company", "directors"])
films = films.merge(wd.add_prefix("wd_").rename(columns={"wd_id": "id"}), on="id", how="left")
assert films["id"].is_unique

## 6. Combined budget and revenue

First non-missing value wins: TMDB → The Numbers → Wikidata. Zero means missing in every source.

In [ ]:
def first_valid(*cols):
    vals = pd.concat([c.where(c > 0) for c, _ in cols], axis=1)
    src = pd.Series(pd.NA, index=vals.index, dtype="string")
    for i, (_, label) in reversed(list(enumerate(cols))):
        src = src.mask(vals.iloc[:, i].notna(), label)
    return vals.bfill(axis=1).iloc[:, 0], src


films["fin_budget"], films["fin_budget_src"] = first_valid(
    (films["budget"], "tmdb"), (films["tn_Production Budget (USD)"], "the_numbers"), (films["wd_wd_budget_usd"], "wikidata"))
films["fin_revenue"], films["fin_revenue_src"] = first_valid(
    (films["revenue"], "tmdb"), (films["tn_Worldwide Gross (USD)"], "the_numbers"), (films["wd_wd_box_office_usd"], "wikidata"))

films["usable"] = (films["year"].between(*YEARS) & films["fin_budget"].ge(BUDGET_FLOOR) & films["fin_revenue"].gt(0))
usable = films[films["usable"]]
print(f"Usable films: {len(usable):,} (TMDB alone: {len(pop):,}, +{len(usable) - len(pop):,})")
pd.crosstab(usable["fin_budget_src"], usable["fin_revenue_src"], margins=True)

## 7. The Numbers films with no TMDB match

They have budget and gross but no TMDB fields (genre, language, etc.). Kept separately for now; whether to use them depends on which inputs the model needs.

In [ ]:
tn_only = tn.drop(index=tn_match["src_row"])
tn_only = tn_only[tn_only["Production Budget (USD)"].ge(BUDGET_FLOOR) & tn_only["Worldwide Gross (USD)"].gt(0)]
print(f"The Numbers films not in TMDB, usable on financials: {len(tn_only):,}")
print(f"Potential total: {len(usable) + len(tn_only):,}")

## 8. Hand-check sample

50 random The Numbers ↔ TMDB matches, weighted toward the riskier ones (year differs or budgets disagree). Review in `data/clean/edwards_tn_handcheck.csv`; mark `correct` as Y/N.

In [ ]:
rng = np.random.default_rng(560)
chk = tn_match.merge(tmdb[["id", "title", "original_title", "year", "runtime", "budget"]], on="id")
chk = chk.join(tn[["Movie Name", "year", "Running Time (minutes)", "Production Budget (USD)"]]
               .add_prefix("tn_"), on="src_row")
risky = chk[chk["year_diff"].gt(0) | ~chk["budget_agrees"]]
sample = pd.concat([risky.sample(25, random_state=rng.integers(1e9)),
                    chk.drop(risky.index).sample(25, random_state=rng.integers(1e9))])
sample["risky"] = sample.index.isin(risky.index)
sample["correct"] = ""
out = CLEAN / "edwards_tn_handcheck.csv"
if not out.exists():                      # written once, never overwritten (same rule as Matt's hand-check)
    sample.drop(columns="src_row").to_csv(out, index=False)
print(f"{len(risky):,} risky matches of {len(chk):,}; sample at {out}")

### 8b. Hand-check result

Reviewed by Tanner, 2026-11-22.

In [ ]:
from scipy.stats import beta

hc = pd.read_csv(CLEAN / "edwards_tn_handcheck.csv")
hc["ok"] = hc["correct"].astype(str).str.strip().str.upper().str.startswith("Y")
assert hc["correct"].notna().all(), "hand-check not finished"

def upper95(wrong, n):          # Clopper-Pearson exact 95% upper bound
    return beta.ppf(0.975, wrong + 1, n - wrong) if wrong < n else 1.0

res = hc.groupby("risky")["ok"].agg(checked="size", correct="sum")
res.loc["all"] = res.sum()
res["wrong"] = res["checked"] - res["correct"]
res["error rate 95% CI upper"] = [f"{upper95(w, n):.1%}" for w, n in zip(res["wrong"], res["checked"])]
res

## 9. Save

In [ ]:
# Keep 2000-2023 films with at least one budget or revenue value from any source
keep = films[films["year"].between(*YEARS) & (films["fin_budget"].notna() | films["fin_revenue"].notna())]
keep.to_csv(CLEAN / "edwards_films_merged.csv", index=False)       # gitignored; rebuilt every run
tn_only.to_csv(CLEAN / "edwards_tn_only.csv", index=False)
print(f"Saved {len(keep):,} TMDB films with financial data ({keep['usable'].sum():,} usable) "
      f"and {len(tn_only):,} The Numbers-only films")